In [84]:
import pandas as pd
import sys
import os

# 1. Získáme aktuální složku, kde leží notebook
# (tj. .../attributes/individual/household_position/notebooks)
current_dir = os.getcwd()

# 2. Musíme se dostat o 4 úrovně výše, do kořene projektu
# Cesta: notebooks -> household_position -> individual -> attributes -> ROOT
project_root = os.path.abspath(os.path.join(current_dir, "../../../.."))

# 3. Přidáme tuto cestu do systému, aby Python viděl balíček 'attributes'
if project_root not in sys.path:
    sys.path.append(project_root)

from attributes.individual.household_position.household_position import (get_household_position_joint_age_gender,
                                                                         read_local_household_composition)
from attributes.marginal_data_reader import read_marginal_data



In [85]:
df_households = read_marginal_data(['population', 'households', 'single_person', 'without_children', 'with_children'],
                                   'households')
df_households = df_households.pivot(index='neighb_code', columns='households', values='count')
df_households

    population  households  single_person  without_children  with_children  \
0        70857       39529          22141              6395           8881   
1        21262       10729           4742              2317           3200   
2        30155       15011           7004              3121           4195   
3        48382       24910          11920              4738           7206   
4        22573       11352           5266              2210           3400   
5         8374        4077           1831               798           1286   
6         9258        4384           1943               854           1410   
7        14212        6633           2706              1575           2106   
8        12782        6177           2505              1375           2084   
9        10284        4845           1817              1036           1811   
10       12078        5836           2366              1274           2000   
11        4617        2114            794               449     

households,households,population,single_person,with_children,without_children
neighb_code,,,,,
550973,39529,70857,22141,8881,6395
550990,10729,21262,4742,3200,2317
551007,15011,30155,7004,4195,3121
551031,24910,48382,11920,7206,4738
551058,11352,22573,5266,3400,2210
551066,4077,8374,1831,1286,798
551074,4384,9258,1943,1410,854
551082,6633,14212,2706,2106,1575
551091,6177,12782,2505,2084,1375


From the first row, it immediately becomes clear that the total reported number of households is not very meaningful:

In [86]:
df_households.loc[550973]

households
households          39529
population          70857
single_person       22141
with_children        8881
without_children     6395
Name: 550973, dtype: int64

The sum of `single_person` = 10, `with_children` = 5 and `without_children` = 5 is 20, while `households` reports a total of 10 households.
This is because of the rounding that is meant to maintain privacy. 
The true number of households is anywhere between 8 and 12, the true number of single person households is also between 8 and 12, and the number of households with or without children is either between 3 and 7.

In this instance, we can see the data cannot be made consistent, by assuming the true number of households is supposed to be 12 (upper bound), with 8 single-person households (lower bound), and 3 with-children and without children households each (again lower bound). 

In [87]:
df_households.loc[:,
'total_households'] = df_households.single_person + df_households.with_children + df_households.without_children
df_households.total_households - df_households.households

neighb_code
550973   -2112
550990    -470
551007    -691
551031   -1046
551058    -476
551066    -162
551074    -177
551082    -246
551091    -213
551112    -181
551147    -196
551171     -81
551198    -355
551210     -13
551228    -117
551236     -91
551244    -246
551252    -103
551279    -176
551287    -379
551295    -222
551309    -115
551317     -71
551325     -43
551368    -102
551376     -23
551406     -18
551422     -10
551431      -4
dtype: int64

Based on the above, let's not look at the total reported number of households at all

In [88]:
df_households.drop(['households', 'total_households'], axis=1, inplace=True)
df_households

households,population,single_person,with_children,without_children
neighb_code,,,,
550973,70857,22141,8881,6395
550990,21262,4742,3200,2317
551007,30155,7004,4195,3121
551031,48382,11920,7206,4738
551058,22573,5266,3400,2210
551066,8374,1831,1286,798
551074,9258,1943,1410,854
551082,14212,2706,2106,1575
551091,12782,2505,2084,1375


The first data set, `df_household_position`, is on the level of individuals, while the second, `df_households` is on the level of households.
If we want to combine the two, we want to find out how many people live in each of the household types `single_person`, `with_children` and `without_children`.

A single-person household by definition contains a single individual.
The semantics of `without_children` in this case seems to indicate married or non-married couples, but also includes households with the designation `other`.
We have no idea how big an `other`-type household could be, so let's assume this just relates to couples without children, in which case this household type is by definition a two-person household type.

That means that in each neighborhood, `single_person` is also the count for number of people living in a single-person household, `without_children` is half the count of people living in a `without_children` household, which means that the rest of the population should be living in a `with_children` household.

Note that `with_children` includes married and non-married couples with children, but also single-parent households

In [89]:
df_households.loc[:, 'in_hh_without_children'] = df_households.without_children * 2
df_households.loc[:,
'in_hh_with_children'] = df_households.population - df_households.single_person - df_households.in_hh_without_children
df_households

households,population,single_person,with_children,without_children,in_hh_without_children,in_hh_with_children
neighb_code,,,,,,
550973,70857,22141,8881,6395,12790,35926
550990,21262,4742,3200,2317,4634,11886
551007,30155,7004,4195,3121,6242,16909
551031,48382,11920,7206,4738,9476,26986
551058,22573,5266,3400,2210,4420,12887
551066,8374,1831,1286,798,1596,4947
551074,9258,1943,1410,854,1708,5607
551082,14212,2706,2106,1575,3150,8356
551091,12782,2505,2084,1375,2750,7527


This means the average number of children per household per neighborhood is the following:

In [90]:
(df_households.in_hh_with_children / df_households.with_children) - 2

neighb_code
550973    2.045265
550990    1.714375
551007    2.030751
551031    1.744935
551058    1.790294
551066    1.846812
551074    1.976596
551082    1.967711
551091    1.611804
551112    1.531198
551147    1.582000
551171    1.702532
551198    1.564231
551210    1.659794
551228    1.673092
551236    1.695770
551244    1.674818
551252    1.927825
551279    1.766966
551287    1.645203
551295    1.686285
551309    2.123316
551317    2.069257
551325    2.096330
551368    1.703575
551376    1.863128
551406    1.724490
551422    1.915888
551431    1.696429
dtype: float64

There are two outliers. The small `BU05181785` neighborhood would have 5 households with children, each with 5 children, which sounds like a lot.
The other interesting household reports no children at all. Let's ignore this for now.

Now let's get back the the household positions:

In [91]:
df_household_position = get_household_position_joint_age_gender()
df_household_position['gender'] = df_household_position['gender'].astype(str).replace({"1": "male", "2": "female"})
df_household_position

,gender,age_group,household_position,count
0,male,0-4,child,10832
1,male,10-14,child,9146
2,male,15-19,child,7097
3,male,20-24,child,4641
4,male,25-29,child,2635
...,...,...,...,...
261,female,70-74,single_parent,776
262,female,75-79,single_parent,700
263,female,80-84,single_parent,471
264,female,85-89,single_parent,355


In [92]:
df_household_position.household_position.unique()

array(['child', 'single', 'non_married_no_children',
       'married_no_children', 'non_married_with_children',
       'married_with_children', 'single_parent'], dtype=object)

The goal is to label each individual with their position in a household, _and_ the type of household they live in.

* `child` => `in_hh_with_children`
* `single` => `single_person`
* `non_married_no_children` => `in_hh_without_children`
* `married_no_children` => `in_hh_without_children`
* `non_married_with_children` => `in_hh_with_children`
* `married_with_children` => `in_hh_with_children`
* `single_parent` => `in_hh_with_children`

In [93]:
df_household_position.set_index('household_position', inplace=True)
df_household_position.loc[:, 'household_type'] = None
df_household_position.loc['child', 'household_type'] = 'in_hh_with_children'
df_household_position.loc['single', 'household_type'] = 'single_person'
df_household_position.loc['non_married_no_children', 'household_type'] = 'in_hh_without_children'
df_household_position.loc['married_no_children', 'household_type'] = 'in_hh_without_children'
df_household_position.loc['non_married_with_children', 'household_type'] = 'in_hh_with_children'
df_household_position.loc['married_with_children', 'household_type'] = 'in_hh_with_children'
df_household_position.loc['single_parent', 'household_type'] = 'in_hh_with_children'
df_household_position.reset_index(inplace=True)
df_household_position

,household_position,gender,age_group,count,household_type
0,child,male,0-4,10832,in_hh_with_children
1,child,male,10-14,9146,in_hh_with_children
2,child,male,15-19,7097,in_hh_with_children
3,child,male,20-24,4641,in_hh_with_children
4,child,male,25-29,2635,in_hh_with_children
...,...,...,...,...,...
261,single_parent,female,70-74,776,in_hh_with_children
262,single_parent,female,75-79,700,in_hh_with_children
263,single_parent,female,80-84,471,in_hh_with_children
264,single_parent,female,85-89,355,in_hh_with_children


In [94]:
df_household_position.household_type.isna().sum()

0

In [95]:
df_households.reset_index()

households,neighb_code,population,single_person,with_children,without_children,in_hh_without_children,in_hh_with_children
0,550973,70857,22141,8881,6395,12790,35926
1,550990,21262,4742,3200,2317,4634,11886
2,551007,30155,7004,4195,3121,6242,16909
3,551031,48382,11920,7206,4738,9476,26986
4,551058,22573,5266,3400,2210,4420,12887
5,551066,8374,1831,1286,798,1596,4947
6,551074,9258,1943,1410,854,1708,5607
7,551082,14212,2706,2106,1575,3150,8356
8,551091,12782,2505,2084,1375,2750,7527
9,551112,10284,1817,1811,1036,2072,6395


In [96]:
df_households.reset_index().melt(id_vars='neighb_code',
                                 value_vars=['single_person', 'in_hh_with_children', 'in_hh_without_children'],
                                 value_name='count',
                                 var_name='household_type').groupby(['neighb_code', 'household_type']).sum()

count
neighb_code household_type               
550973      in_hh_with_children     35926
            in_hh_without_children  12790
            single_person           22141
550990      in_hh_with_children     11886
            in_hh_without_children   4634
...                                   ...
551422      in_hh_without_children    124
            single_person              64
551431      in_hh_with_children       621
            in_hh_without_children    182
            single_person             117

[87 rows x 1 columns]

In [97]:
df_households_with_position = df_household_position.merge(
        df_households.melt(
                value_vars=['single_person', 'in_hh_with_children', 'in_hh_without_children'],
                value_name='count',
                var_name='household_type').groupby('household_type').sum(),
        on='household_type',
        how='left')
df_households_with_position

,household_position,gender,age_group,count_x,household_type,count_y
0,child,male,0-4,10832,in_hh_with_children,231432
1,child,male,10-14,9146,in_hh_with_children,231432
2,child,male,15-19,7097,in_hh_with_children,231432
3,child,male,20-24,4641,in_hh_with_children,231432
4,child,male,25-29,2635,in_hh_with_children,231432
...,...,...,...,...,...,...
261,single_parent,female,70-74,776,in_hh_with_children,231432
262,single_parent,female,75-79,700,in_hh_with_children,231432
263,single_parent,female,80-84,471,in_hh_with_children,231432
264,single_parent,female,85-89,355,in_hh_with_children,231432


Now we can adjust the `count_y`, with the frequencies in `count_x`

In [98]:
df_households_with_position.loc[:, 'count'] = df_households_with_position.groupby(
        'household_type').count_x.transform(
        lambda x: x / x.sum() * df_households_with_position.count_y)
df_households_with_position

,household_position,gender,age_group,count_x,household_type,count_y,count
0,child,male,0-4,10832,in_hh_with_children,231432,12794.533968
1,child,male,10-14,9146,in_hh_with_children,231432,10803.065701
2,child,male,15-19,7097,in_hh_with_children,231432,8382.829355
3,child,male,20-24,4641,in_hh_with_children,231432,5481.853042
4,child,male,25-29,2635,in_hh_with_children,231432,3112.407405
...,...,...,...,...,...,...,...
261,single_parent,female,70-74,776,in_hh_with_children,231432,916.595122
262,single_parent,female,75-79,700,in_hh_with_children,231432,826.825496
263,single_parent,female,80-84,471,in_hh_with_children,231432,556.335441
264,single_parent,female,85-89,355,in_hh_with_children,231432,419.318645


In [99]:
df_households_with_position.groupby('household_type')['count'].sum().reset_index()

,household_type,count
0,in_hh_with_children,231432.0
1,in_hh_without_children,79088.0
2,single_person,87990.0


That looks good

In [100]:
df_households_with_position = df_households_with_position[
    ["age_group", "gender", "household_position", "household_type", "count"]]
df_households_with_position

,age_group,gender,household_position,household_type,count
0,0-4,male,child,in_hh_with_children,12794.533968
1,10-14,male,child,in_hh_with_children,10803.065701
2,15-19,male,child,in_hh_with_children,8382.829355
3,20-24,male,child,in_hh_with_children,5481.853042
4,25-29,male,child,in_hh_with_children,3112.407405
...,...,...,...,...,...
261,70-74,female,single_parent,in_hh_with_children,916.595122
262,75-79,female,single_parent,in_hh_with_children,826.825496
263,80-84,female,single_parent,in_hh_with_children,556.335441
264,85-89,female,single_parent,in_hh_with_children,419.318645


# Number of children in a household

Now for our next trick:

## Finding the percentages

In [101]:
df_household_composition = read_local_household_composition()
df_household_composition

typ_domacnosti,reference_person_age,single,ostatni,couple_1_children,couple_2_children,couple_3_children,couple_0_children,single_parent_1_children,single_parent_2_children,single_parent_3_children
0,15-19,683,1241,1973,5410,2068,172,1883,2050,526
1,20-24,4445,4555,2322,2807,901,3327,1836,1229,263
2,25-29,9148,5306,4048,1880,396,8935,1792,712,126
3,30-34,8651,3169,7103,4631,760,6865,1890,649,147
4,35-39,6850,2032,5839,8043,1751,3056,2060,951,209
5,40-44,6208,1580,4999,9812,2377,1954,2675,1395,268
6,45-49,6014,1530,5151,7641,1751,2410,3033,1350,191
7,50-54,5603,1334,4479,3194,677,3801,2377,612,68
8,55-59,6322,1340,3654,1356,235,6968,1966,330,16
9,60-64,5933,1200,2336,580,66,8841,1360,174,11


### Single Parents
$f(h_c)$ = \frac{h_c}{\sum^{c+}_{c=1} h_c}$ 

In [102]:
df_n_children_per_single_parent = df_household_composition.melt(
        value_vars=['single_parent_1_children', 'single_parent_2_children', 'single_parent_3_children'],
        value_name='count',
        var_name='household_type').groupby('household_type').sum().transform(lambda x: x / x.sum())
df_n_children_per_single_parent

,count
household_type,
single_parent_1_children,0.684338
single_parent_2_children,0.265715
single_parent_3_children,0.049948


This suggests about 61% of single-parents has only one child, 28% has two and the remaining 11% has 3 (or more, but we ignore that) children.

This also suggests that 61% of children who live in a single-parent household are the only child in that household, but since a 2-children household houses two children, the remaining fractions change a bit


In [103]:
df_n_children_in_single_parent_household = df_n_children_per_single_parent.copy()
df_n_children_in_single_parent_household.loc[:, 'n_children'] = [1, 2, 3]
df_n_children_in_single_parent_household.loc[:, 'count'] *= df_n_children_in_single_parent_household.n_children
df_n_children_in_single_parent_household.loc[:, 'count'] = df_n_children_in_single_parent_household['count'].transform(
        lambda x: x / x.sum())
df_n_children_in_single_parent_household.drop('n_children', axis=1, inplace=True)
df_n_children_in_single_parent_household

,count
household_type,
single_parent_1_children,0.501123
single_parent_2_children,0.389152
single_parent_3_children,0.109726


The equation can be simplified a bit

$n_c = n \cdot \frac{c \cdot h_c}{\sum^{c^+}_{c'=1} c' \cdot  h_{c'} $
 

In [104]:
df_tst = df_household_composition.melt(
        value_vars=['single_parent_1_children', 'single_parent_2_children', 'single_parent_3_children'],
        value_name='count',
        var_name='household_type').groupby('household_type').sum()
df_tst.loc[:, 'c'] = [1, 2, 3]
df_tst.loc[:, 'f_n_c'] = df_tst.c * df_tst['count']
df_tst.loc[:, 'n_c'] = df_tst['f_n_c'].transform(lambda x: x / x.sum())

df_tst

,count,c,f_n_c,n_c
household_type,,,,
single_parent_1_children,25443,1,25443,0.501123
single_parent_2_children,9879,2,19758,0.389152
single_parent_3_children,1857,3,5571,0.109726


### Couples with children
We can do a similar trick for couples with children:

In [105]:
df_n_children_per_couple = df_household_composition.melt(
        value_vars=['couple_1_children', 'couple_2_children', 'couple_3_children'],
        value_name='count',
        var_name='household_type').groupby('household_type').sum().transform(lambda x: x / x.sum())
df_n_children_per_couple

,count
household_type,
couple_1_children,0.448719
couple_2_children,0.444438
couple_3_children,0.106843


In [106]:
df_n_children_in_couple_household = df_n_children_per_couple.copy()
df_n_children_in_couple_household.loc[:, 'n_children'] = [1, 2, 3]
df_n_children_in_couple_household.loc[:, 'count'] *= df_n_children_in_couple_household.n_children
df_n_children_in_couple_household.loc[:, 'count'] = df_n_children_in_couple_household['count'].transform(
        lambda x: x / x.sum())
df_n_children_in_couple_household.drop('n_children', axis=1, inplace=True)
df_n_children_in_couple_household

,count
household_type,
couple_1_children,0.270619
couple_2_children,0.536074
couple_3_children,0.193307


### Children
For children, it is not yet known if they live in a single-parent or two-parent household. This means we do not have to split them into three groups (1, 2 or 3+ children) but in six groups (1, 2, 3+ children in a single-parent or two-parent household)

For this, we have to know the relative frequencies of single- and two-parent households as well.

In [107]:
df_single_vs_couple_distribution = df_households_with_position[df_households_with_position.household_position.isin(
        ['married_with_children', 'non_married_with_children', 'single_parent'])].groupby('household_position')[
    "count"].sum().transform(lambda x: x / x.sum())
df_single_vs_couple_distribution

household_position
married_with_children        0.598010
non_married_with_children    0.200828
single_parent                0.201162
Name: count, dtype: float64

So now we know that 60%% of children has two married parents and of that 60%, 22% lives in a single-child household. We can combine those two facts:

In [108]:
df_n_children_per_couple

,count
household_type,
couple_1_children,0.448719
couple_2_children,0.444438
couple_3_children,0.106843


In [109]:
df_n_children = pd.concat([
    df_n_children_in_single_parent_household.rename(index={
        'single_parent_1_children': 'child_of_single_parent_1_children',
        'single_parent_2_children': 'child_of_single_parent_2_children',
        'single_parent_3_children': 'child_of_single_parent_3_children'
    }),
    df_n_children_in_couple_household.rename(index={
        'couple_1_children': 'child_in_married_with_1_children',
        'couple_2_children': 'child_in_married_with_2_children',
        'couple_3_children': 'child_in_married_with_3_children'
    }),
    df_n_children_in_couple_household.rename(index={
        'couple_1_children': 'child_in_non_married_with_1_children',
        'couple_2_children': 'child_in_non_married_with_2_children',
        'couple_3_children': 'child_in_non_married_with_3_children'
    }),
])
msk_single_parent = df_n_children.index.str.contains('single_parent')
msk_non_married = df_n_children.index.str.contains('non_married')
df_n_children.loc[msk_single_parent, 'count'] *= df_single_vs_couple_distribution.loc['single_parent']
df_n_children.loc[msk_non_married, 'count'] *= df_single_vs_couple_distribution.loc['non_married_with_children']
df_n_children.loc[~(msk_single_parent | msk_non_married), 'count'] *= df_single_vs_couple_distribution.loc[
    'married_with_children']
df_n_children

,count
household_type,
child_of_single_parent_1_children,0.100807
child_of_single_parent_2_children,0.078283
child_of_single_parent_3_children,0.022073
child_in_married_with_1_children,0.161833
child_in_married_with_2_children,0.320577
child_in_married_with_3_children,0.115600
child_in_non_married_with_1_children,0.054348
child_in_non_married_with_2_children,0.107658
child_in_non_married_with_3_children,0.038821


In [110]:
df_n_children.sum()

count    1.0
dtype: float64

Actually, we have to do the same for the `df_n_children_per_couple` frame, splitting the couples with children into married or unmarried

In [111]:
df_married_vs_not_distribution = df_households_with_position[df_households_with_position.household_position.isin(
        ['married_with_children', 'non_married_with_children'])].groupby('household_position')[
    "count"].sum().transform(lambda x: x / x.sum())
df_married_vs_not_distribution

household_position
married_with_children        0.7486
non_married_with_children    0.2514
Name: count, dtype: float64

In [112]:
df_n_couples = pd.concat([
    df_n_children_per_couple.rename(index={
        'couple_1_children': 'married_with_1_children', 'couple_2_children': 'married_with_2_children',
        'couple_3_children': 'married_with_3_children'
    }),
    df_n_children_per_couple.rename(index={
        'couple_1_children': 'non_married_with_1_children', 'couple_2_children': 'non_married_with_2_children',
        'couple_3_children': 'non_married_with_3_children'
    })])
msk = df_n_couples.index.str.contains('non_married')
df_n_couples.loc[msk, 'count'] *= df_married_vs_not_distribution.loc['non_married_with_children']
df_n_couples.loc[~msk, 'count'] *= df_married_vs_not_distribution.loc['married_with_children']
df_n_couples

,count
household_type,
married_with_1_children,0.335912
married_with_2_children,0.332706
married_with_3_children,0.079982
non_married_with_1_children,0.112808
non_married_with_2_children,0.111732
non_married_with_3_children,0.026860


In [113]:
df_n_couples.sum()

count    1.0
dtype: float64

## Using these frequencies

### Children

In [114]:
df_households_with_position_children = df_households_with_position[
    df_households_with_position.household_position == 'child'].assign(key=1).merge(
        df_n_children.reset_index().assign(key=1), on='key').drop('key', axis=1)
df_households_with_position_children

,age_group,gender,household_position,household_type_x,count_x,household_type_y,count_y
0,0-4,male,child,in_hh_with_children,12794.533968,child_of_single_parent_1_children,0.100807
1,0-4,male,child,in_hh_with_children,12794.533968,child_of_single_parent_2_children,0.078283
2,0-4,male,child,in_hh_with_children,12794.533968,child_of_single_parent_3_children,0.022073
3,0-4,male,child,in_hh_with_children,12794.533968,child_in_married_with_1_children,0.161833
4,0-4,male,child,in_hh_with_children,12794.533968,child_in_married_with_2_children,0.320577
...,...,...,...,...,...,...,...
337,90+,female,child,in_hh_with_children,0.000000,child_in_married_with_2_children,0.320577
338,90+,female,child,in_hh_with_children,0.000000,child_in_married_with_3_children,0.115600
339,90+,female,child,in_hh_with_children,0.000000,child_in_non_married_with_1_children,0.054348
340,90+,female,child,in_hh_with_children,0.000000,child_in_non_married_with_2_children,0.107658


### Single Parents


In [115]:
df_households_with_position_single_parent = df_households_with_position[
    df_households_with_position.household_position == 'single_parent'].assign(key=1).merge(
        df_n_children_in_single_parent_household.reset_index().assign(key=1), on='key').drop('key', axis=1)
df_households_with_position_single_parent

,age_group,gender,household_position,household_type_x,count_x,household_type_y,count_y
0,0-4,male,single_parent,in_hh_with_children,0.000000,single_parent_1_children,0.501123
1,0-4,male,single_parent,in_hh_with_children,0.000000,single_parent_2_children,0.389152
2,0-4,male,single_parent,in_hh_with_children,0.000000,single_parent_3_children,0.109726
3,10-14,male,single_parent,in_hh_with_children,0.000000,single_parent_1_children,0.501123
4,10-14,male,single_parent,in_hh_with_children,0.000000,single_parent_2_children,0.389152
...,...,...,...,...,...,...,...
109,85-89,female,single_parent,in_hh_with_children,419.318645,single_parent_2_children,0.389152
110,85-89,female,single_parent,in_hh_with_children,419.318645,single_parent_3_children,0.109726
111,90+,female,single_parent,in_hh_with_children,301.200717,single_parent_1_children,0.501123
112,90+,female,single_parent,in_hh_with_children,301.200717,single_parent_2_children,0.389152


### Married couples

In [116]:
df_households_with_position_married_parents = df_households_with_position[
    df_households_with_position.household_position == 'married_with_children'].assign(key=1).merge(
        df_n_children_in_couple_household.rename(index={
            'couple_1_children': 'married_with_1_children',
            'couple_2_children': 'married_with_2_children',
            'couple_3_children': 'married_with_3_children',
        }).reset_index().assign(key=1), on='key').drop('key', axis=1)
df_households_with_position_married_parents

,age_group,gender,household_position,household_type_x,count_x,household_type_y,count_y
0,0-4,male,married_with_children,in_hh_with_children,0.000000,married_with_1_children,0.270619
1,0-4,male,married_with_children,in_hh_with_children,0.000000,married_with_2_children,0.536074
2,0-4,male,married_with_children,in_hh_with_children,0.000000,married_with_3_children,0.193307
3,10-14,male,married_with_children,in_hh_with_children,0.000000,married_with_1_children,0.270619
4,10-14,male,married_with_children,in_hh_with_children,0.000000,married_with_2_children,0.536074
...,...,...,...,...,...,...,...
109,85-89,female,married_with_children,in_hh_with_children,38.978916,married_with_2_children,0.536074
110,85-89,female,married_with_children,in_hh_with_children,38.978916,married_with_3_children,0.193307
111,90+,female,married_with_children,in_hh_with_children,8.268255,married_with_1_children,0.270619
112,90+,female,married_with_children,in_hh_with_children,8.268255,married_with_2_children,0.536074


In [117]:
df_households_with_position_non_married_parents = df_households_with_position[
    df_households_with_position.household_position == 'non_married_with_children'].assign(key=1).merge(
        df_n_children_in_couple_household.rename(index={
            'couple_1_children': 'non_married_with_1_children',
            'couple_2_children': 'non_married_with_2_children',
            'couple_3_children': 'non_married_with_3_children',
        }).reset_index().assign(key=1), on='key').drop('key', axis=1)
df_households_with_position_non_married_parents

,age_group,gender,household_position,household_type_x,count_x,household_type_y,count_y
0,0-4,male,non_married_with_children,in_hh_with_children,0.0,non_married_with_1_children,0.270619
1,0-4,male,non_married_with_children,in_hh_with_children,0.0,non_married_with_2_children,0.536074
2,0-4,male,non_married_with_children,in_hh_with_children,0.0,non_married_with_3_children,0.193307
3,10-14,male,non_married_with_children,in_hh_with_children,0.0,non_married_with_1_children,0.270619
4,10-14,male,non_married_with_children,in_hh_with_children,0.0,non_married_with_2_children,0.536074
...,...,...,...,...,...,...,...
109,85-89,female,non_married_with_children,in_hh_with_children,0.0,non_married_with_2_children,0.536074
110,85-89,female,non_married_with_children,in_hh_with_children,0.0,non_married_with_3_children,0.193307
111,90+,female,non_married_with_children,in_hh_with_children,0.0,non_married_with_1_children,0.270619
112,90+,female,non_married_with_children,in_hh_with_children,0.0,non_married_with_2_children,0.536074


## Bringing it back together

In [118]:
df_households_with_children = pd.concat([
    df_households_with_position_children,
    df_households_with_position_single_parent,
    df_households_with_position_married_parents,
    df_households_with_position_non_married_parents
])
df_households_with_children.loc[:, 'count'] = df_households_with_children.count_x * df_households_with_children.count_y
df_households_with_children.loc[:, 'household_position'] = df_households_with_children.household_type_y
df_households_with_children.rename(columns={'household_type_x': 'household_type'}, inplace=True)
df_households_with_children = df_households_with_children[
    ['age_group', 'gender', 'household_position', 'household_type', 'count']]
df_households_with_children

,age_group,gender,household_position,household_type,count
0,0-4,male,child_of_single_parent_1_children,in_hh_with_children,1289.779376
1,0-4,male,child_of_single_parent_2_children,in_hh_with_children,1001.590257
2,0-4,male,child_of_single_parent_3_children,in_hh_with_children,282.410129
3,0-4,male,child_in_married_with_1_children,in_hh_with_children,2070.575422
4,0-4,male,child_in_married_with_2_children,in_hh_with_children,4101.638587
...,...,...,...,...,...
109,85-89,female,non_married_with_2_children,in_hh_with_children,0.000000
110,85-89,female,non_married_with_3_children,in_hh_with_children,0.000000
111,90+,female,non_married_with_1_children,in_hh_with_children,0.000000
112,90+,female,non_married_with_2_children,in_hh_with_children,0.000000


In [119]:
df_households_with_position_and_children = pd.concat([
    df_households_with_position[
        df_households_with_position.household_position.isin(['single', 'non_married_no_children',
                                                             'married_no_children'])],
    df_households_with_children
])
df_households_with_position_and_children

,age_group,gender,household_position,household_type,count
38,0-4,male,single,single_person,0.0
39,10-14,male,single,single_person,0.0
40,15-19,male,single,single_person,336.0
41,20-24,male,single,single_person,2340.0
42,25-29,male,single,single_person,5551.0
...,...,...,...,...,...
109,85-89,female,non_married_with_2_children,in_hh_with_children,0.0
110,85-89,female,non_married_with_3_children,in_hh_with_children,0.0
111,90+,female,non_married_with_1_children,in_hh_with_children,0.0
112,90+,female,non_married_with_2_children,in_hh_with_children,0.0


In [120]:
df_households_with_position['count'].sum()

398510.0

In [121]:
df_households_with_position_and_children['count'].sum()

398510.0

In [122]:
pd.concat([
    df_households_with_position.groupby('household_type')['count'].sum(),
    df_households_with_position_and_children.groupby('household_type')['count'].sum()
], axis=1)

,count,count
household_type,,
in_hh_with_children,231432.0,231432.0
in_hh_without_children,79088.0,79088.0
single_person,87990.0,87990.0


The two data frames still look the same. We can start adding household position from the `df_households_with_position_and_children` frame

In [123]:
df_households_with_position_and_children.to_pickle('../processed/df_households_with_position_and_children.pkl')

In [124]:
print(df_households_with_position_and_children)

    age_group  gender           household_position       household_type  \
38        0-4    male                       single        single_person   
39      10-14    male                       single        single_person   
40      15-19    male                       single        single_person   
41      20-24    male                       single        single_person   
42      25-29    male                       single        single_person   
..        ...     ...                          ...                  ...   
109     85-89  female  non_married_with_2_children  in_hh_with_children   
110     85-89  female  non_married_with_3_children  in_hh_with_children   
111       90+  female  non_married_with_1_children  in_hh_with_children   
112       90+  female  non_married_with_2_children  in_hh_with_children   
113       90+  female  non_married_with_3_children  in_hh_with_children   

      count  
38      0.0  
39      0.0  
40    336.0  
41   2340.0  
42   5551.0  
..      ...  
1